# 레슨 16 - Microsoft Foundry로 확장 가능한 에이전트 배포하기

이 노트북에서는 가상의 회사 <strong>Contoso</strong>를 위한 <strong>프로덕션 준비된 고객 지원 에이전트</strong>를 구축합니다. 이전 레슨과 달리 여기서의 핵심은 에이전트의 추론 루프가 아니라 — 에이전트를 안전하게 대규모로 운영할 수 있게 하는 <em>그것을 둘러싼 모든 것</em>입니다:

1. **도구 호출** — 주문 조회 및 티켓 생성.
2. **RAG** — 지식 베이스에서 정책 답변 제공.
3. <strong>메모리</strong> — 대화 흐름을 통해 고객 기억하기.
4. **모델 라우팅** — 간단한 요청은 작은 모델로, 복잡한 요청은 큰 모델로 전송.
5. **응답 캐싱** — 반복되는 질문은 모델 호출 없이 제공.
6. **사람 승인** — 특정 금액 이상의 환불은 승인 대기.
7. **평가 게이트** — 나쁜 릴리스를 차단하는 오프라인 테스트 세트.
8. <strong>관찰성</strong> — 모든 요청에 OpenTelemetry 추적 적용.

각 섹션은 독립적이며 실행 가능합니다. 모든 줄을 읽어보세요 — 프로덕션 기본 요소들은 의도적으로 작게 유지되어 있습니다.


## 설정

이 노트북을 실행하기 전에 다음을 확인하세요:

1. **배포된 챗 모델이 있는 Microsoft Foundry 프로젝트** (예: `gpt-4.1-mini`).
2. **Azure CLI에 로그인** — 터미널에서 `az login`을 실행하세요.
3. **필요한 환경 변수를 설정**:
   - `AZURE_AI_PROJECT_ENDPOINT` — Microsoft Foundry 프로젝트 엔드포인트.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — 배포된 모델 이름.

RAG 섹션은 `AZURE_SEARCH_SERVICE_ENDPOINT`와 `AZURE_SEARCH_API_KEY`가 설정되어 있을 때 <strong>Azure AI Search</strong>를 사용하며, Search 리소스 없이도 노트북이 실행되도록 메모리 내 검색으로 대체됩니다.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import re
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential(),
)
print("Foundry client ready.")

## 1. 도구

기존 도구는 실제 시스템에 대해 실제 작업을 수행합니다. 여기서는 일반 Python 함수를 사용하여 주문 데이터베이스와 티켓팅 시스템을 시뮬레이션합니다. `@tool` 데코레이터는 이들을 에이전트에 노출합니다.

`issue_refund`가 일정 금액 이상의 환불에 대해 `approval_mode="always_require"`를 사용하는 점에 주목하세요 — 이것은 나중에 배포할 인간 개입 방식의 원시 기능입니다.


In [ ]:
# Simulated backend systems (in production these are API calls behind scoped identities).
ORDERS = {
    "A1001": {"status": "shipped", "total": 42.00, "eta": "2 days"},
    "A1002": {"status": "processing", "total": 128.50, "eta": "5 days"},
    "A1003": {"status": "delivered", "total": 19.99, "eta": "delivered"},
}
TICKETS: list[dict] = []
REFUND_APPROVAL_THRESHOLD = 50.0


@tool(approval_mode="never_require")
def get_order_status(order_id: Annotated[str, "The customer's order ID, e.g. A1001"]) -> str:
    """Look up the status of a customer order."""
    order = ORDERS.get(order_id.upper())
    if not order:
        return f"No order found with ID {order_id}."
    return (
        f"Order {order_id.upper()}: status={order['status']}, "
        f"total=${order['total']:.2f}, eta={order['eta']}."
    )


@tool(approval_mode="never_require")
def open_ticket(
    subject: Annotated[str, "Short subject line for the support ticket"],
    details: Annotated[str, "Full description of the customer's issue"],
) -> str:
    """Open a support ticket for issues that need human follow-up."""
    ticket_id = f"T{1000 + len(TICKETS) + 1}"
    TICKETS.append({"id": ticket_id, "subject": subject, "details": details})
    return f"Ticket {ticket_id} opened: {subject}"


def refund_needs_approval(amount: float) -> bool:
    """Refunds above the threshold require a human approver."""
    return amount > REFUND_APPROVAL_THRESHOLD


@tool(approval_mode="always_require")
def issue_refund(
    order_id: Annotated[str, "The order to refund"],
    amount: Annotated[float, "Refund amount in USD"],
) -> str:
    """Issue a refund. Execution pauses for human approval before it runs."""
    return f"Refund of ${amount:.2f} issued for order {order_id.upper()}."


print("Tools defined.")

## 2. RAG — 정책 지식 베이스

정책 질문("반품 기간이 어떻게 되나요?")은 모델의 기억이 아닌 권위 있는 출처에서 답변되어야 합니다. 우리는 작은 지식 베이스를 검색 도구로 래핑합니다.

실제 운용에서는 <strong>Azure AI Search</strong>를 사용하지만, 여기서는 노트북이 어디서든 실행될 수 있도록 메모리 내 키워드 검색을 제공하며 환경 변수에 따라 자동으로 Azure AI Search로 전환합니다.


In [ ]:
KNOWLEDGE_BASE = {
    "returns": "Contoso accepts returns within 30 days of delivery for a full refund. Items must be unused and in original packaging.",
    "shipping": "Standard shipping takes 3-5 business days. Express shipping (1-2 days) is available at checkout for an extra fee.",
    "warranty": "All Contoso electronics carry a 12-month limited warranty covering manufacturing defects.",
    "refund_policy": "Refunds are processed to the original payment method within 5 business days of approval. Refunds over $50 require a supervisor's approval.",
}


def _in_memory_search(query: str) -> str:
    q = query.lower()
    hits = [text for key, text in KNOWLEDGE_BASE.items() if key.replace("_", " ") in q or key in q]
    if not hits:
        # crude keyword fallback so the tool still returns something useful
        hits = [text for text in KNOWLEDGE_BASE.values() if any(w in text.lower() for w in q.split())]
    return "\n".join(hits) if hits else "No matching policy found."


def _azure_search(query: str) -> str:
    from azure.core.credentials import AzureKeyCredential
    from azure.search.documents import SearchClient

    client = SearchClient(
        endpoint=os.environ["AZURE_SEARCH_SERVICE_ENDPOINT"],
        index_name=os.getenv("AZURE_SEARCH_INDEX_NAME", "contoso-policies"),
        credential=AzureKeyCredential(os.environ["AZURE_SEARCH_API_KEY"]),
    )
    results = client.search(search_text=query, top=3)
    return "\n".join(r.get("content", "") for r in results) or "No matching policy found."


USE_AZURE_SEARCH = bool(os.getenv("AZURE_SEARCH_SERVICE_ENDPOINT") and os.getenv("AZURE_SEARCH_API_KEY"))


@tool(approval_mode="never_require")
def search_policies(query: Annotated[str, "The policy question to look up"]) -> str:
    """Search Contoso support policies to answer customer questions."""
    if USE_AZURE_SEARCH:
        return _azure_search(query)
    return _in_memory_search(query)


print(f"RAG ready. Using {'Azure AI Search' if USE_AZURE_SEARCH else 'in-memory search'}.")

## 3. 메모리

자신이 누구와 대화하고 있는지 잊어버리는 지원 에이전트는 좋은 지원 에이전트가 아닙니다. 우리는 고객별로 작은 프로필 저장소를 유지하고 요약된 내용을 에이전트 지침에 주입합니다. 프로덕션 환경에서는 이것이 메모리 서비스입니다 (레슨 13 참조); 여기서는 dict가 패턴을 보여줍니다.


In [ ]:
CUSTOMER_MEMORY: dict[str, dict] = {
    "cust-42": {"name": "Dana", "tier": "enterprise", "recent_order": "A1002"},
    "cust-99": {"name": "Sam", "tier": "standard", "recent_order": "A1003"},
}


def memory_context(customer_id: str) -> str:
    profile = CUSTOMER_MEMORY.get(customer_id)
    if not profile:
        return "This is a new customer with no history."
    return (
        f"Customer {profile['name']} ({profile['tier']} tier). "
        f"Most recent order: {profile['recent_order']}."
    )


print(memory_context("cust-42"))

## 4 & 5. 모델 라우팅과 응답 캐싱

단일 요청 핸들러에 연결된 두 가지 비용 조절 장치:

- <strong>라우팅</strong>: 저렴한 휴리스틱 분류기가 요청이 작은 모델을 필요로 하는지 큰 모델을 필요로 하는지 결정합니다.
- <strong>캐싱</strong>: 정규화된 반복 질문은 모델 호출 없이 캐시에서 바로 제공됩니다.

여기서 분류기는 의도적으로 단순합니다. 운영 환경에서는 트래픽을 기준으로 검증하고 Foundry의 모델 라우터로 대체할 수 있습니다.


In [ ]:
SMALL_MODEL = os.getenv("AZURE_AI_SMALL_MODEL", model)   # e.g. gpt-4.1-mini
LARGE_MODEL = os.getenv("AZURE_AI_LARGE_MODEL", model)   # e.g. gpt-4.1

response_cache: dict[str, str] = {}
route_counters = {"small": 0, "large": 0, "cache": 0}


def normalize(query: str) -> str:
    return re.sub(r"\s+", " ", query.lower().strip())


COMPLEX_SIGNALS = ("refund", "cancel", "complaint", "escalate", "broken", "wrong", "why")


def is_simple(query: str) -> bool:
    """Route complex or high-stakes requests to the large model; everything else to the small one."""
    q = query.lower()
    if any(signal in q for signal in COMPLEX_SIGNALS):
        return False
    return len(q.split()) <= 20


def choose_model(query: str) -> str:
    return SMALL_MODEL if is_simple(query) else LARGE_MODEL


print(f"Small model: {SMALL_MODEL} | Large model: {LARGE_MODEL}")

## 6 & 8. 에이전트, 인간 승인 및 관측성

이제 위의 도구들로 에이전트를 조립하고 각 요청을 OpenTelemetry 스팬으로 래핑합니다. `handle_support_request` 함수는 프로덕션 요청 처리기입니다: 캐시 → 라우팅 → 추적 → 실행 → 캐시.

인간 승인은 프레임워크에서 처리합니다: `issue_refund`가 `approval_mode="always_require"`이기 때문에 실행이 일시 중지되고 명시적으로 해결하는 승인 요청이 표시됩니다.


In [ ]:
# Tracing: use the Agent Framework tracer if available, else a no-op so the notebook runs anywhere.
try:
    from agent_framework.observability import get_tracer
    tracer = get_tracer()
except Exception:  # observability extras not installed
    from contextlib import contextmanager

    class _NoopSpan:
        def set_attribute(self, *_args, **_kwargs):
            pass

    class _NoopTracer:
        @contextmanager
        def start_as_current_span(self, _name):
            yield _NoopSpan()

    tracer = _NoopTracer()


SUPPORT_INSTRUCTIONS = (
    "You are Contoso's customer support agent. Be concise, friendly, and accurate. "
    "Use search_policies for policy questions, get_order_status for orders, "
    "open_ticket when a human needs to follow up, and issue_refund for refunds. "
    "Never invent policy details."
)

support_agent = provider.as_agent(
    name="ContosoSupportAgent",
    instructions=SUPPORT_INSTRUCTIONS,
    tools=[get_order_status, open_ticket, search_policies, issue_refund],
)


async def handle_support_request(query: str, customer_id: str) -> str:
    # 1. Serve from cache when we can.
    key = normalize(query)
    if key in response_cache:
        route_counters["cache"] += 1
        return response_cache[key]

    # 2. Route by complexity to control cost.
    chosen_model = choose_model(query)
    route_counters["small" if chosen_model == SMALL_MODEL else "large"] += 1

    # 3. Add per-customer memory to the prompt.
    context = memory_context(customer_id)
    prompt = f"[Customer context: {context}]\n\n{query}"

    # 4. Run inside a trace span for observability.
    with tracer.start_as_current_span("support_request") as span:
        span.set_attribute("customer.id", customer_id)
        span.set_attribute("routed.model", chosen_model)
        response = await support_agent.run(prompt, model=chosen_model)

    text = response.text
    response_cache[key] = text
    return text


print("Support agent assembled.")

In [ ]:
# Try a few requests. The first is simple (small model), the second is a refund (large model + approval path).
print(await handle_support_request("What is your return window?", "cust-99"))
print("---")
print(await handle_support_request("Where is my order A1002?", "cust-42"))
print("---")
# Repeat the first question -> served from cache.
print(await handle_support_request("What is your return window?", "cust-99"))
print("---")
print("Routing counters:", route_counters)

## 7. 평가 게이트

이것은 수업의 릴리스 게이트입니다: 오프라인 테스트 세트가 에이전트를 평가하고, 합격률이 임계값을 넘겨야만 배포가 진행됩니다. 여기서 평가자는 노트북을 자체 포함하도록 간단한 키워드 중복 확인 방식입니다; 실제 운영 환경에서는 LLM-판사나 프레임워크 평가자를 사용할 것입니다(수업 10 참고).


In [ ]:
TEST_CASES = [
    {"input": "How long do I have to return an item?", "expected": ["30 days", "refund"]},
    {"input": "How fast is standard shipping?", "expected": ["3-5", "business days"]},
    {"input": "What is the status of order A1001?", "expected": ["shipped", "A1001"]},
    {"input": "Do your electronics have a warranty?", "expected": ["12-month", "warranty"]},
]


def score_response(actual: str, expected_keywords: list[str]) -> float:
    actual_l = actual.lower()
    hits = sum(1 for kw in expected_keywords if kw.lower() in actual_l)
    return hits / len(expected_keywords)


async def evaluation_gate(test_cases: list[dict], threshold: float = 0.8) -> bool:
    passed = 0
    for case in test_cases:
        result = await support_agent.run(case["input"])
        s = score_response(result.text, case["expected"])
        status = "PASS" if s >= 0.5 else "FAIL"
        print(f"[{status}] {case['input']}  (score={s:.0%})")
        if s >= 0.5:
            passed += 1
    pass_rate = passed / len(test_cases)
    print(f"\nEvaluation pass rate: {pass_rate:.0%} (gate: {threshold:.0%})")
    return pass_rate >= threshold


gate_passed = await evaluation_gate(TEST_CASES, threshold=0.8)
print("\nDeploy allowed:" , gate_passed)

## 함께 구성하기: 시뮬레이션된 릴리스

아래 셀은 수업에서 설명하는 전체 루프를 보여줍니다: 평가 게이트를 실행하고 통과할 경우에만 "배포"합니다. 이것이 에이전트 버전을 Foundry Agent Service에 프로모션하기 전에 CI에서 실행하는 패턴입니다.


In [ ]:
async def release(test_cases: list[dict]) -> None:
    print("Running pre-deployment evaluation gate...\n")
    if await evaluation_gate(test_cases, threshold=0.8):
        print("\n✅ Gate passed — promoting agent version to the Foundry Agent Service.")
    else:
        print("\n❌ Gate failed — release blocked. Fix the agent and re-run.")


await release(TEST_CASES)

## 요약

모든 운영 문제를 연결한 프로덕션 준비가 된 고객 지원 에이전트를 조립했습니다:

- <strong>도구, RAG, 메모리</strong>는 에이전트에 기능과 맥락을 제공합니다.
- <strong>모델 라우팅 및 캐싱</strong>은 지연 시간과 비용을 통제합니다.
- <strong>사람 승인</strong>은 대규모 환불과 같은 고위험 작업을 보호합니다.
- <strong>평가 게이트</strong>는 잘못된 출시를 차단합니다.
- <strong>추적</strong>은 모든 요청을 관찰할 수 있게 합니다.

### 과제

이 에이전트를 다음과 같이 확장하세요:

1. **여러 모델 지원** — 세 번째 "추론" 레벨을 추가하고 에스컬레이션/불만을 해당 레벨로 라우팅합니다.
2. **평가 게이트 추가** — `TEST_CASES`를 확장하여 환불 승인 시나리오를 포함하고 게이트가 리그레션을 잡는지 확인합니다.
3. **비용 인식 라우팅 추가** — 요청당 예상 비용(소형 대 대형 대 캐시)을 추적하고 혼합 쿼리 배치 후 비용 보고서를 출력합니다.

다음 수업에서는 정반대 여정을 거쳐 Microsoft Foundry Local 및 Qwen과 함께 완전히 자신의 머신에서 에이전트를 실행합니다.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**면책 조항**:
이 문서는 AI 번역 서비스 [Co-op Translator](https://github.com/Azure/co-op-translator)를 사용하여 번역되었습니다. 정확성을 기하기 위해 노력하고 있으나, 자동 번역은 오류나 부정확한 부분이 있을 수 있음을 유의하시기 바랍니다. 원본 문서의 원어본이 권위 있는 자료로 간주되어야 합니다. 중요한 정보의 경우, 전문가의 인간 번역을 권장합니다. 이 번역 사용으로 인해 발생하는 오해나 잘못된 해석에 대해 당사는 책임을 지지 않습니다.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
